In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score,classification_report, confusion_matrix

In [5]:
df = pd.read_csv('shop_smart_ecommerce.csv')
X = df.drop(columns=['Revenue'])
y = df['Revenue'].astype(int)

print(X.shape)
print(y.shape)

(12330, 17)
(12330,)


In [6]:
# find numerical and categorical columns 
num_features = X.select_dtypes(include=['int64', 'float64']).columns
cat_features = X.select_dtypes(exclude=['int64', 'float64']).columns

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y, stratify=y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown="ignore"), cat_features)
    ]
)

In [8]:
dt = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=30,
    class_weight='balanced',
    random_state=42
)

In [9]:
pipe = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', dt)
    ]
)

print('Pipeline created')

Pipeline created


In [11]:
pipe.fit(X_train,y_train)

y_pred = pipe.predict(X_test)

print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

F1 Score: 0.6278

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.85      0.90      2084
           1       0.50      0.83      0.63       382

    accuracy                           0.85      2466
   macro avg       0.73      0.84      0.77      2466
weighted avg       0.89      0.85      0.86      2466


Confusion Matrix:
 [[1771  313]
 [  64  318]]


In [13]:
from sklearn.model_selection import GridSearchCV

#
param_grid = {
    "model__max_depth": [4, 6, 8, 10],
    "model__min_samples_leaf": [20, 30, 40, 50]
}


grid = GridSearchCV(
    estimator=pipe,        
    param_grid=param_grid, 
    scoring="f1",         
    cv=5,                 
    n_jobs=-1              
)

print("GridSearchCV testing all combinations...")


grid.fit(X_train, y_train)


print("\n--- GridSearchCV Final Results ---")
print(f"Best F1 Score (Cross-Validated): {grid.best_score_:.4f}")
print("Best Parameters:", grid.best_params_)

GridSearchCV testing all combinations...

--- GridSearchCV Final Results ---
Best F1 Score (Cross-Validated): 0.6344
Best Parameters: {'model__max_depth': 4, 'model__min_samples_leaf': 50}


In [ ]:
final_predictions = grid.predict(X_test)

print("Final Optimized Model Evaluation ")
print(f"Test F1 Score: {f1_score(y_test, final_predictions):.4f}")
print("\nClassification Report:\n", classification_report(y_test, final_predictions))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, final_predictions))

Final Optimized Model Evaluation 
Test F1 Score: 0.6229

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.85      0.90      2084
           1       0.50      0.83      0.62       382

    accuracy                           0.84      2466
   macro avg       0.73      0.84      0.76      2466
weighted avg       0.89      0.84      0.86      2466


Confusion Matrix:
 [[1763  321]
 [  64  318]]
